# Oil & Gas — Forecasting (H6.S2)
This notebook compares simple **Naive**, optional **ARIMA** (`statsmodels`) and optional **Prophet** (`prophet`) on an annual oil series.

**Inputs**
- `docs/eda/oil_yearly_series.csv` (from H6.S1) — must contain a `year` column and an oil value column (e.g., `oil_bbl`).

**Outputs** (written to `docs/forecast/`)
- `metrics.csv` — MAE / RMSE per model
- `predictions.csv` — test-window predictions
- `model_choice.json` — best model by RMSE
- `future_forecast.csv` — 3-year simple forecast using the best model

In [11]:
import os
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Config
INPUT_CSV = "../docs/eda/oil_yearly_series.csv"
OUTDIR = "../docs/forecast"
TEST_YEARS = 2
FUTURE_YEARS = 3
VALUE_CANDIDATES = ["oil_bbl", "oil", "value", "y"]
print("Config:", INPUT_CSV, OUTDIR, TEST_YEARS, FUTURE_YEARS)

os.makedirs(OUTDIR, exist_ok=True)

# Load data
df = pd.read_csv(INPUT_CSV)

# Use directly if the CSV already has columns ds,y
if set(["ds","y"]).issubset(df.columns):
    d = df[["ds","y"]].dropna().copy()
    d["ds"] = pd.to_datetime(d["ds"])  # ensure date format
else:
    raise ValueError("CSV must have 'ds' and 'y' columns")

display(d.tail())


Config: ../docs/eda/oil_yearly_series.csv ../docs/forecast 2 3


,ds,y
12,2013-01-01,313080.0
13,2014-01-01,355325.0
14,2015-01-01,279462.0
15,2016-01-01,220901.0
16,2017-01-01,192679.0


In [12]:
# Train/Test split (last N years = test)
d = d.sort_values("ds")
cutoff_idx = len(d) - TEST_YEARS
if cutoff_idx < 5:
    raise ValueError("Not enough history for a test split; reduce TEST_YEARS.")
train = d.iloc[:cutoff_idx].reset_index(drop=True)
test = d.iloc[cutoff_idx:].reset_index(drop=True)
print("Train range:", train["ds"].min().date(), "→", train["ds"].max().date(), "| n=", len(train))
print("Test  range:", test["ds"].min().date(), "→", test["ds"].max().date(), "| n=", len(test))

Train range: 2001-01-01 → 2015-01-01 | n= 15
Test  range: 2016-01-01 → 2017-01-01 | n= 2


In [ ]:
# Helpers
def mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))

results = []
preds_rows = []  # collect test predictions

### Naive (last observed value)

In [14]:
last_value = float(train['y'].iloc[-1])
naive_pred = np.full(len(test), last_value, dtype=float)
m_mae = mae(test['y'], naive_pred)
m_rmse = rmse(test['y'], naive_pred)
results.append({"model":"naive_last", "mae":m_mae, "rmse":m_rmse})
preds_rows += [{"model":"naive_last","ds":ds, "y_true":yt, "y_pred":yp} for ds, yt, yp in zip(test["ds"], test["y"], naive_pred)]
print("Naive - MAE:", m_mae, "RMSE:", m_rmse)

Naive - MAE: 72672.0 RMSE: 74029.31787474474


### ARIMA (optional, statsmodels)

In [15]:
try:
    import itertools
    from statsmodels.tsa.arima.model import ARIMA

    # Simple grid search over small set of (p,d,q)
    p = [0,1,2]
    d_vals = [0,1,2]
    q = [0,1,2]
    best = None
    for order in itertools.product(p, d_vals, q):
        try:
            model = ARIMA(train['y'], order=order)
            fitted = model.fit()
            fc = fitted.forecast(steps=len(test))
            m_mae = mae(test['y'], fc)
            m_rmse = rmse(test['y'], fc)
            cand = (m_rmse, order, fc)
            if best is None or cand < best:
                best = cand
        except Exception:
            continue
    if best:
        m_rmse, order, fc = best
        m_mae = mae(test['y'], fc)
        results.append({"model":f"arima{order}", "mae":m_mae, "rmse":m_rmse})
        preds_rows += [{"model":f"arima{order}","ds":ds, "y_true":yt, "y_pred":yp} for ds, yt, yp in zip(test["ds"], test["y"], fc)]
        print("ARIMA", order, "- MAE:", m_mae, "RMSE:", m_rmse)
    else:
        print("ARIMA: no valid order converged; skipping.")
except Exception as e:
    print("ARIMA not available or failed:", e)

ARIMA (0, 0, 2) - MAE: 28321.832159474594 RMSE: 29675.67407966989


### Prophet (optional)

In [16]:
try:
    from prophet import Prophet
    pf = Prophet()
except Exception as e:
    pf = None
    print("Prophet not available:", e)

if pf is not None:
    try:
        pf = Prophet(growth='linear', yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
        pf.fit(train.rename(columns={"ds":"ds","y":"y"}))
        fc = pf.predict(pd.DataFrame({"ds": test["ds"]}))["yhat"].values
        m_mae = mae(test["y"], fc)
        m_rmse = rmse(test["y"], fc)
        results.append({"model":"prophet", "mae":m_mae, "rmse":m_rmse})
        preds_rows += [{"model":"prophet","ds":ds, "y_true":yt, "y_pred":yp} for ds, yt, yp in zip(test["ds"], test["y"], fc)]
        print("Prophet - MAE:", m_mae, "RMSE:", m_rmse)
    except Exception as e:
        print("Prophet failed:", e)

15:57:12 - cmdstanpy - INFO - Chain [1] start processing


15:57:12 - cmdstanpy - INFO - Chain [1] done processing


Prophet - MAE: 226034.16206791505 RMSE: 227216.13987276587


In [17]:
# Save metrics & predictions
metrics = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
preds = pd.DataFrame(preds_rows)
metrics_path = os.path.join(OUTDIR, "metrics.csv")
preds_path = os.path.join(OUTDIR, "predictions.csv")
metrics.to_csv(metrics_path, index=False)
preds.to_csv(preds_path, index=False)
print("Saved:", metrics_path, "|", preds_path)
display(metrics)

Saved: ../docs/forecast/metrics.csv | ../docs/forecast/predictions.csv


,model,mae,rmse
0,"arima(0, 0, 2)",28321.832159,29675.674080
1,naive_last,72672.000000,74029.317875
2,prophet,226034.162068,227216.139873


In [ ]:
# Select best model by RMSE and produce a simple 3-year forecast
if not len(metrics):
    raise RuntimeError("No models produced metrics.")

best_model = metrics.iloc[0]["model"]
choice = {"selected_model": best_model, "timestamp": pd.Timestamp.utcnow().isoformat()}
with open(os.path.join(OUTDIR, "model_choice.json"), "w") as f:
    json.dump(choice, f, indent=2)
print("Selected:", best_model)

# Simple future forecasting per model choice
last_ds = d["ds"].max()
future_dates = pd.date_range(last_ds + pd.offsets.DateOffset(years=1), periods=FUTURE_YEARS, freq="YS")

if best_model.startswith("naive"):
    future_vals = np.full(len(future_dates), float(d["y"].iloc[-1]))
elif best_model.startswith("arima"):
    from statsmodels.tsa.arima.model import ARIMA
    order = eval(best_model.replace("arima",""))
    fitted = ARIMA(d["y"], order=order).fit()
    future_vals = fitted.forecast(steps=len(future_dates)).values
elif best_model == "prophet":
    from prophet import Prophet
    pf = Prophet(growth='linear', yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
    pf.fit(d.rename(columns={"ds":"ds","y":"y"}))
    future_vals = pf.predict(pd.DataFrame({"ds": future_dates}))["yhat"].values
else:
    future_vals = np.full(len(future_dates), float(d["y"].iloc[-1]))

future = pd.DataFrame({"ds": future_dates, "yhat": future_vals})
future_path = os.path.join(OUTDIR, "future_forecast.csv")
future.to_csv(future_path, index=False)
print("Saved:", future_path)
future.tail()

Selected: arima(0, 0, 2)
Saved: ../docs/forecast/future_forecast.csv


,ds,yhat
0,2018-01-01,244988.406046
1,2019-01-01,262332.109151
2,2020-01-01,271083.215288
